# Phase 2 — Generalization and Data Augmentation Effects

Reproduces **Methods 2.4.2** and **Results 3.2** of *Automated Segmentation and Morphometric Assessment of Neuronal Somata in the Rat Trigeminal Ganglion Using a Fine-Tuned Cellpose-SAM Model*.

The configuration selected in Phase 1 (`n_epochs = 100`, `lr = 1e-5`, `wd = 0.1`, `batch_size = 1`) is evaluated by 5-fold cross-validation (`KFold(shuffle=True, random_state=42)`) over the 26 annotated images (train + test folders), under two conditions:

- **A** — without augmentation
- **B** — with augmentation, applied only to the training images of each fold (original + 2 augmented versions per image): horizontal flip, vertical flip and 90° rotation (p = 0.5 each); affine transform, scale [0.95, 1.05], rotation [−20°, 20°] (p = 0.5); brightness/contrast ±10% (p = 0.3); elastic transform (p = 0.15)

Total: 2 conditions × 5 folds = 10 models. Inference: `diameter = None`, `channel_axis = -1`, other parameters at Cellpose defaults.

**Notebook series** (run in this order; each notebook reads the outputs of the previous ones):

| # | Notebook | Manuscript |
|---|----------|------------|
| 1 | `Phase1_training_hyperparameter_exploration.ipynb` | Methods 2.4.1 · Results 3.1 |
| 2 | `Phase2_generalization_data_augmentation.ipynb` | Methods 2.4.2 · Results 3.2 |
| 3 | `Phase3_inference_parameter_optimization.ipynb` | Methods 2.4.3 · Results 3.3 |
| 4 | `Phase4_morphology_measurements.ipynb` | Methods 2.5 · Results 3.4 · Table 1 · Figures 8–9 |
| 5 | `Phase5_figures.ipynb` | Figures 2–7 (no GPU needed) |

## Required files

| Path variable | Content | Provided file |
|---------------|---------|---------------|
| `TRAIN_DIR` | 21 images (`<id>.tif`) and annotations (`<id>_seg.npy`) | `dataset/train/` |
| `TEST_DIR` | 5 images (`1`, `7`, `10`, `15`, `17`) and annotations | `dataset/test/` |

The 26 images are pooled in the order *sorted train names + sorted test names*; this order defines the folds and must not be changed.

## Outputs

| File | Content |
|------|---------|
| `MODELS_DIR/<condition>/<condition>_fold_<k>_ep100_lr1e-05_wd0.1.pth` | 10 fine-tuned models (inputs of Phases 3 and 4) |
| `RESULTS_DIR/<condition>/results_final.json` | Per-fold metrics, test image names, loss curves (input of Phase 3) |
| `RESULTS_DIR/<condition>/folds_test_info.json` | Test images of each fold |
| `RESULTS_DIR/<condition>/metrics/`, `per_image_metrics/` | Per-model files |
| `RESULTS_DIR/phase2_all_results.csv` | One row per model (input of `Phase5_figures`, Figure 5) |
| `RESULTS_DIR/phase2_per_image_consolidated.csv` | Per-image metrics of all models |
| `RESULTS_DIR/phase2_final_summary.json` | Mean ± SD per condition |

`RESULTS_DIR = PROJECT_DIR/results/phase2`, `MODELS_DIR = PROJECT_DIR/models`.

**Expected folder layout** (files are distributed separately from the notebooks; set `PROJECT_DIR` in the *Paths* cell to the folder that contains them):

```
PROJECT_DIR/
├── dataset/
│   ├── train/        21 images <id>.tif + Cellpose annotations <id>_seg.npy
│   ├── test/          5 images (1, 7, 10, 15, 17) + <id>_seg.npy
│   └── validation/    5 images (3.2.2, Valid1–Valid4) + <id>_seg.npy
├── models/
│   ├── A_no_augmentation/     A_no_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
│   └── B_with_augmentation/   B_with_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
├── results/
│   ├── phase1/  phase2/  phase3/  phase4/
└── figures/
```

**Environment used in the study:** Google Colab Pro, NVIDIA RTX PRO 6000 Blackwell Server Edition GPU, Python 3, `cellpose==4.2.1.1` (Cellpose-SAM), PyTorch 2.11 (CUDA 12.8). Fixed seed: `RANDOM_SEED = 42`.

## 1. Environment

In [ ]:
!pip install --quiet "cellpose==4.2.1.1" albumentations

In [ ]:
import gc
import json
import time
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold

import cellpose
from cellpose import io, models, train
from cellpose.metrics import average_precision

print(f"cellpose {cellpose.version} | torch {torch.__version__} | albumentations {A.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Paths

Only this cell needs to be edited. On Google Colab, mount Google Drive first (`from google.colab import drive; drive.mount('/content/drive')`) and point `PROJECT_DIR` to the folder that contains `dataset/`, `models/` and `results/`.

In [ ]:
PROJECT_DIR = Path(".")

TRAIN_DIR = PROJECT_DIR / "dataset" / "train"
TEST_DIR = PROJECT_DIR / "dataset" / "test"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2"

## 3. Parameters (fixed by the study design)

In [ ]:
N_EPOCHS = 100
LEARNING_RATE = 0.00001
WEIGHT_DECAY = 0.1
BATCH_SIZE = 1
AUGMENTATION_MULTIPLIER = 2
K_FOLDS = 5
IOU_THRESHOLDS = [0.5, 0.75, 0.9]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

EXPERIMENTS = ["A_no_augmentation", "B_with_augmentation"]
for exp in EXPERIMENTS:
    (MODELS_DIR / exp).mkdir(parents=True, exist_ok=True)
    for sub in ("metrics", "per_image_metrics"):
        (RESULTS_DIR / exp / sub).mkdir(parents=True, exist_ok=True)

## 4. Data loading

In [ ]:
def load_image_and_mask(img_path, mask_path):
    img = io.imread(str(img_path))
    if img.ndim == 3 and img.shape[-1] == 4:
        img = img[:, :, :3]
    mask = io.imread(str(mask_path))
    return img, mask


def load_dataset(folder_path):
    folder = Path(folder_path)
    img_files = sorted([
        f for f in folder.glob("*")
        if f.suffix.lower() in [".tif", ".tiff", ".png", ".jpg"]
        and "_mask" not in f.stem
        and "_masks" not in f.stem
        and "_seg" not in f.stem
    ])

    images, masks, names = [], [], []
    for img_file in img_files:
        candidates = [
            folder / f"{img_file.stem}_masks.tif",
            folder / f"{img_file.stem}_masks.png",
            folder / f"{img_file.stem}_mask.tif",
            folder / f"{img_file.stem}_seg.npy",
        ]
        mask_path = next((p for p in candidates if p.exists()), None)
        if mask_path is None:
            print(f"WARNING: no mask for {img_file.name}")
            continue

        if mask_path.suffix == ".npy":
            mask = np.load(str(mask_path), allow_pickle=True).item()["masks"]
            img = io.imread(str(img_file))
            if img.ndim == 3 and img.shape[-1] == 4:
                img = img[:, :, :3]
        else:
            img, mask = load_image_and_mask(img_file, mask_path)

        images.append(img)
        masks.append(mask)
        names.append(img_file.stem)

    return images, masks, names


train_images, train_masks, train_names = load_dataset(TRAIN_DIR)
test_images, test_masks, test_names = load_dataset(TEST_DIR)

all_images = train_images + test_images
all_masks = train_masks + test_masks
all_names = train_names + test_names

print(f"Total: {len(all_images)} images")
print(f"Order: {all_names}")

## 5. Data augmentation

In [ ]:
def get_augmentation_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.95, 1.05), rotate=(-20, 20), border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.ElasticTransform(alpha=20, sigma=5, p=0.15),
    ])


def augment_data(train_data, train_labels, augmentation_multiplier=2):
    transform = get_augmentation_pipeline()
    augmented_images, augmented_masks = [], []
    for img, mask in zip(train_data, train_labels):
        augmented_images.append(img)
        augmented_masks.append(mask)
        for _ in range(augmentation_multiplier):
            augmented = transform(image=img, mask=mask)
            augmented_images.append(augmented["image"])
            augmented_masks.append(augmented["mask"])
    return augmented_images, augmented_masks

## 6. Evaluation functions

Object-level metrics from `cellpose.metrics.average_precision` at IoU 0.50, 0.75 and 0.90: AP = TP / (TP + FP + FN), Precision = TP / (TP + FP), Recall = TP / (TP + FN), F1 = 2PR / (P + R). Aggregate AP is the mean of the per-image AP values.

In [ ]:
def compute_metrics_at_threshold(test_labels, predicted_masks, threshold):
    ap, tp, fp, fn = average_precision(test_labels, predicted_masks, threshold=[threshold])

    ap_per_image = ap.flatten().tolist()
    tp_per_image = tp.flatten().astype(int).tolist()
    fp_per_image = fp.flatten().astype(int).tolist()
    fn_per_image = fn.flatten().astype(int).tolist()

    total_tp, total_fp, total_fn = int(np.sum(tp)), int(np.sum(fp)), int(np.sum(fn))
    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

    precision_per_image = [t / (t + f + 1e-8) for t, f in zip(tp_per_image, fp_per_image)]
    recall_per_image = [t / (t + f + 1e-8) for t, f in zip(tp_per_image, fn_per_image)]
    f1_per_image = [2 * (p * r) / (p + r + 1e-8) for p, r in zip(precision_per_image, recall_per_image)]

    return {
        "aggregate": {
            "AP": float(np.mean(ap)), "TP": total_tp, "FP": total_fp, "FN": total_fn,
            "Precision": float(precision), "Recall": float(recall), "F1": float(f1),
        },
        "per_image": {
            "AP": ap_per_image, "TP": tp_per_image, "FP": fp_per_image, "FN": fn_per_image,
            "Precision": precision_per_image, "Recall": recall_per_image, "F1": f1_per_image,
        },
    }


def evaluate_model(model, test_data, test_labels, test_names_list):
    predicted_masks, inference_times = [], []
    for img in test_data:
        t0 = time.time()
        masks_pred, _, _ = model.eval(img, diameter=None, channel_axis=-1)
        inference_times.append(time.time() - t0)
        predicted_masks.append(masks_pred)

    results = {
        "thresholds": {},
        "per_image_thresholds": {},
        "inference_time_s": inference_times,
        "test_image_names": test_names_list,
    }
    for thr in IOU_THRESHOLDS:
        metrics = compute_metrics_at_threshold(test_labels, predicted_masks, thr)
        results["thresholds"][f"AP@{thr}"] = metrics["aggregate"]
        results["per_image_thresholds"][f"AP@{thr}"] = metrics["per_image"]
    return results


def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 7. K-fold training

In [ ]:
def train_kfold_experiment(experiment_name, use_augmentation):
    print(f"\n{'#' * 70}\n# {experiment_name} | augmentation = {use_augmentation}\n{'#' * 70}")
    exp_results_dir = RESULTS_DIR / experiment_name
    exp_models_dir = MODELS_DIR / experiment_name

    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    fold_results, folds_test_info = [], []

    for fold, (train_idx, test_idx) in enumerate(kf.split(all_images), start=1):
        train_data_original = [all_images[i] for i in train_idx]
        train_labels_original = [all_masks[i] for i in train_idx]
        test_data = [all_images[i] for i in test_idx]
        test_labels = [all_masks[i] for i in test_idx]
        test_names_fold = [all_names[i] for i in test_idx]

        if use_augmentation:
            train_data, train_labels = augment_data(train_data_original, train_labels_original,
                                                    augmentation_multiplier=AUGMENTATION_MULTIPLIER)
        else:
            train_data, train_labels = train_data_original, train_labels_original

        print(f"\nFOLD {fold}/{K_FOLDS} | train = {len(train_data)} | test = {test_names_fold}")
        folds_test_info.append({"fold": fold, "test_image_names": test_names_fold,
                                "test_indices": test_idx.tolist(),
                                "n_train": len(train_data), "n_test": len(test_data)})

        free_gpu_memory()
        base_model = models.CellposeModel(gpu=True)
        model_name = f"{experiment_name}_fold_{fold}_ep{N_EPOCHS}_lr{LEARNING_RATE}_wd{WEIGHT_DECAY}"

        start = time.time()
        _, train_losses, _ = train.train_seg(
            base_model.net,
            train_data=train_data,
            train_labels=train_labels,
            test_data=None,
            test_labels=None,
            batch_size=BATCH_SIZE,
            n_epochs=N_EPOCHS,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            nimg_per_epoch=len(train_data),
            model_name=model_name,
            save_every=N_EPOCHS,
        )
        train_time = time.time() - start

        final_model_path = exp_models_dir / f"{model_name}.pth"
        base_model.net.save_model(str(final_model_path))

        eval_start = time.time()
        eval_results = evaluate_model(base_model, test_data, test_labels, test_names_fold)
        eval_time = time.time() - eval_start

        fold_result = {
            "fold": fold,
            "experiment": experiment_name,
            "model_name": model_name,
            "model_file": final_model_path.name,
            "use_augmentation": use_augmentation,
            "hyperparameters": {
                "n_epochs": N_EPOCHS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "batch_size": BATCH_SIZE,
                "augmentation_multiplier": AUGMENTATION_MULTIPLIER if use_augmentation else 0,
            },
            "n_train": len(train_data),
            "n_train_original": len(train_data_original),
            "n_test": len(test_data),
            "test_image_names": test_names_fold,
            "training_time_seconds": train_time,
            "evaluation_time_seconds": eval_time,
            "train_loss_final": float(train_losses[-1]) if len(train_losses) > 0 else None,
            "train_losses_curve": [float(x) for x in train_losses],
            "aggregate_metrics": eval_results["thresholds"],
            "per_image_metrics": eval_results["per_image_thresholds"],
            "inference_time_per_image_s": eval_results["inference_time_s"],
        }
        fold_results.append(fold_result)

        with open(exp_results_dir / "metrics" / f"{model_name}_metrics.json", "w") as f:
            json.dump(fold_result, f, indent=4)

        records = []
        for i, img_name in enumerate(test_names_fold):
            record = {"image_name": img_name, "experiment": experiment_name,
                      "fold": fold, "use_augmentation": use_augmentation}
            for thr in IOU_THRESHOLDS:
                key = f"AP@{thr}"
                per_image = eval_results["per_image_thresholds"][key]
                for m in ("AP", "TP", "FP", "FN", "Precision", "Recall", "F1"):
                    record[f"{key}_{m}"] = per_image[m][i]
            records.append(record)
        pd.DataFrame(records).to_csv(exp_results_dir / "per_image_metrics" / f"{model_name}_per_image.csv", index=False)

        agg = eval_results["thresholds"]
        print(f"AP@0.5 = {agg['AP@0.5']['AP']:.4f} | AP@0.75 = {agg['AP@0.75']['AP']:.4f} | "
              f"AP@0.9 = {agg['AP@0.9']['AP']:.4f} | F1@0.5 = {agg['AP@0.5']['F1']:.4f} | "
              f"training time = {train_time:.1f}s")

        del base_model
        free_gpu_memory()

        with open(exp_results_dir / "results_partial.json", "w") as f:
            json.dump(fold_results, f, indent=4)

    with open(exp_results_dir / "results_final.json", "w") as f:
        json.dump(fold_results, f, indent=4)
    with open(exp_results_dir / "folds_test_info.json", "w") as f:
        json.dump(folds_test_info, f, indent=4)
    return fold_results

## 8. Condition A — without augmentation

In [ ]:
train_kfold_experiment("A_no_augmentation", use_augmentation=False)

## 9. Condition B — with augmentation

In [ ]:
train_kfold_experiment("B_with_augmentation", use_augmentation=True)

## 10. Consolidation

In [ ]:
def results_to_dataframe(results_list):
    records = []
    for r in results_list:
        record = {
            "fold": r["fold"],
            "experiment": r["experiment"],
            "use_augmentation": r["use_augmentation"],
            "model_name": r["model_name"],
            "n_train": r["n_train"],
            "n_test": r["n_test"],
            "training_time_s": r["training_time_seconds"],
            "train_loss_final": r["train_loss_final"],
        }
        for thr in IOU_THRESHOLDS:
            key = f"AP@{thr}"
            for m in ("AP", "TP", "FP", "FN", "Precision", "Recall", "F1"):
                record[f"{key}_{m}"] = r["aggregate_metrics"][key][m]
        records.append(record)
    return pd.DataFrame(records)


def load_results(experiment):
    with open(RESULTS_DIR / experiment / "results_final.json") as f:
        return json.load(f)


df_A = results_to_dataframe(load_results("A_no_augmentation"))
df_B = results_to_dataframe(load_results("B_with_augmentation"))
df_all = pd.concat([df_A, df_B], ignore_index=True)
df_all.to_csv(RESULTS_DIR / "phase2_all_results.csv", index=False)

per_image = pd.concat(
    [pd.read_csv(f) for exp in EXPERIMENTS
     for f in sorted((RESULTS_DIR / exp / "per_image_metrics").glob("*_per_image.csv"))],
    ignore_index=True,
).sort_values(["experiment", "fold", "image_name"]).reset_index(drop=True)
per_image.to_csv(RESULTS_DIR / "phase2_per_image_consolidated.csv", index=False)

summary = {
    "config_used": {"n_epochs": N_EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
                    "batch_size": BATCH_SIZE, "augmentation_multiplier_when_used": AUGMENTATION_MULTIPLIER},
}
for label, df in (("A_no_augmentation", df_A), ("B_with_augmentation", df_B)):
    summary[label] = {f"{col}_{stat}": float(getattr(df[col], stat)())
                      for col in ("AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP",
                                  "AP@0.5_Precision", "AP@0.5_Recall", "AP@0.5_F1", "training_time_s")
                      for stat in ("mean", "std")}
with open(RESULTS_DIR / "phase2_final_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

for label, df in (("A — without augmentation", df_A), ("B — with augmentation", df_B)):
    print(label)
    for col in ("AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP", "AP@0.5_Precision", "AP@0.5_Recall", "AP@0.5_F1"):
        print(f"  {col:18s} {df[col].mean():.3f} ± {df[col].std():.3f}")